# M6 lab: putting carbon in the model

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M6 · about 45 minutes, plus 90 minutes of practice at the end.

You will rebuild the lesson's planning LP for the Plains Grid, solve it with the open-source HiGHS solver through `scipy.optimize.linprog(method="highs")`, and check your results against the lesson's reference block with `assert` cells:

1. The six days that stand for 2025, and the costs.
2. One LP: what to build (wind, solar, a 4-hour battery) and how to run every hour, at least yearly cost.
3. A CO₂ cap and its dual; the frontier traced by seven caps.
4. A carbon price instead of a cap, and the ties a price can leave.
5. A 300 MW data center (a flat load) with annual or hourly matching rules: the cost, the claim and what the plants emit.
6. The full year: two of the plans checked on all 8,760 hours.

**How to run it** (about a minute):

- Put this notebook and the four data files in one folder.
- **In Google Colab:** *File → Upload notebook*, upload the four files with the folder icon on the left, then *Runtime → Run all*.
- **Locally:** JupyterLab, or VS Code with its Jupyter extension, and Python 3.9 or later with `numpy` and `scipy` 1.9 or later. Start it from that folder: the files are read from the working folder.

| File (download from the lesson page) | What it holds |
|---|---|
| `a10-emissions-questions.json` | the Plains Grid's fleet (block `plains`): sizes, EIA's 2025 heat rates, gas price and CO₂ coefficient |
| `m06-carbon-in-models.json` | what may be built and its costs, the six days and their weights, the caps, the load, and the reference results |
| `town_year.json` | the 8,760 hourly shapes of 2025 (Southwest Power Pool, EIA-930), in thousandths of each series' maximum |
| `cf_targets.json` | EIA's 2025 capacity factors, annual and monthly (wind 33.8%, solar 24.0%) |

**The data:**

- The Plains Grid is fictional; its demand, wind and sun follow real 2025 hourly shapes.
- Gas plants use EIA's 2025 heat rates (7,535 and 10,975 Btu per kWh), gas at \$3.85 per MMBtu and 52.91 kg CO₂ per MMBtu.
- New wind and solar cost what EIA's AEO2026 gives; the 4-hour battery what NREL's 2025 update gives, moved from 2024 to 2025 dollars with the GDP deflator.
- Money is in 2025 dollars, discounted at a 4% real rate (inflation taken out).

**Ties:**

- An LP can have many best plans. The asserts check only numbers that every best plan shares: costs, CO₂, sizes.
- Section 4 shows how to find a range when they differ.
- A schedule (which plant is curtailed, how a battery cycles, which hours go short) can differ between best plans even when those numbers agree.

In [ ]:
import json
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
FILES = ["a10-emissions-questions.json", "m06-carbon-in-models.json", "town_year.json", "cf_targets.json"]
for name in FILES:
    assert Path(name).exists(), f"{name} not found: put it next to this notebook (in Colab, upload it with the folder icon)."
A10, M6, year, targets = (json.load(open(n, encoding="utf-8")) for n in FILES)
P, REF = A10["plains"], M6["reference"]
print("NumPy", np.__version__, "· SciPy", scipy.__version__)

## 1. The six days and the costs

- **Wind and solar** follow the 2025 shapes, scaled as the lesson's `profiles.js` does (method "scale"): each month first to EIA's monthly capacity factor, then the year to 33.8% (wind) and 24.0% (solar): output $= \min(1, k\,s_t)$ per MW of plant.
- **The days**: each season's medoid (the day closest, in summed squares of demand, wind and solar, to the season's other days), plus the calmest day and the day of the highest net load, each weighted by the days it stands for (365 in all).
- **Hours**: index $i$ of `town_year.json` is the hour ending at `start_utc` + $i$; a day starts at 00:00 Central time (daylight time from 9 March to 1 November).

In [ ]:
def unit(a):
    return np.maximum(np.asarray(a, float), 0) / 1000

def k_for(s, cf):
    # k with mean(min(1, k s)) = cf, by bisection, as profiles.js cfScale(method "scale")
    f = lambda k: np.minimum(1, k * s).mean()
    lo, hi = 0.0, 1.0
    while f(hi) < cf:
        hi *= 2
    for _ in range(200):
        if hi - lo <= 1e-13 * max(1, hi):
            break
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if f(mid) < cf else (lo, mid)
    return (lo + hi) / 2

def per_mw(tech):
    s = unit(year[tech]); days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]; out = s.copy(); t0 = 0
    for m, d in enumerate(days):
        seg = s[t0:t0 + 24 * d]; out[t0:t0 + 24 * d] = np.minimum(1, k_for(seg, targets["monthly"][tech][m]) * seg); t0 += 24 * d
    return np.minimum(1, k_for(out, targets["targets"][tech]["cf"]) * out)

PW, PS = per_mw("wind"), per_mw("solar")
print(f"capacity factors over 2025: wind {PW.mean():.4f}, solar {PS.mean():.4f}")
assert abs(PW.mean() - 0.338) < 1e-9 and abs(PS.mean() - 0.24) < 1e-9

def js_round(x):   # JavaScript's Math.round (halves up), so whole-MW values match the page
    return np.floor(np.asarray(x) + 0.5)

from datetime import date
def start_index(d):
    off = -5 if "2025-03-09" <= d < "2025-11-02" else -6
    return (date.fromisoformat(d) - date(2025, 1, 1)).days * 24 - 7 + 1 - off

def day_of(d):
    i0 = start_index(d); sl = slice(i0, i0 + 24)
    return {"demand": js_round(np.asarray(year["demand"][sl]) / 1000 * P["demandPeakMW"]),
            "we": js_round(PW[sl] * P["windMW"]), "se": js_round(PS[sl] * P["solarMW"]), "cfw": PW[sl], "cfs": PS[sl]}

DAYS = []
for x in M6["days"]:
    d = day_of(x["date"]); d.update(weight=x["weight"], label=x["label"], date=x["date"]); DAYS.append(d)
    print(f"{x['date']}  {x['label']:34s} weight {x['weight']:3d}  demand {d['demand'].sum():6.0f} MWh  wind CF {d['cfw'].mean():.3f}")
assert sum(d["weight"] for d in DAYS) == 365
NH = 24 * len(DAYS); WGT = np.repeat([d["weight"] for d in DAYS], 24)
DEM = np.concatenate([d["demand"] for d in DAYS]); WE = np.concatenate([d["we"] for d in DAYS]); SE = np.concatenate([d["se"] for d in DAYS])
CFW = np.concatenate([d["cfw"] for d in DAYS]); CFS = np.concatenate([d["cfs"] for d in DAYS])

def crf(r, n):
    return r / (1 - (1 + r) ** -n)
fin, B = M6["finance"], M6["build"]
defl = M6["deflator"]["2025"] / M6["deflator"]["2024"]
bat_kw = (B["battery"]["powerCapex2024"] + 4 * B["battery"]["energyCapex2024"]) * defl
CAP = {"W": B["wind"]["capex"] * 1000 * crf(fin["rate"], fin["life"]), "S": B["solar"]["capex"] * 1000 * crf(fin["rate"], fin["life"]),
       "B": bat_kw * 1000 * crf(fin["rate"], fin["batteryLife"])}
FOM = {"W": B["wind"]["fom"] * 1000, "S": B["solar"]["fom"] * 1000, "B": B["battery"]["fomShare"] * bat_kw * 1000}
GAS = {k: {"mw": p["mw"], "rate": p["heatRate"] * P["gas"]["co2PerMMBtu"] / 1000, "cost": p["heatRate"] * P["gas"]["price"] + p["vom"]} for k, p in P["plants"].items()}
print({k: round(CAP[k] + FOM[k]) for k in CAP}, "$ per MW a year;", {k: (round(v["rate"], 4), round(v["cost"], 2)) for k, v in GAS.items()})

## 2. One LP for the year

Columns (the decision variables): the three new sizes $W, S, B$ (MW), then for each of the 144 hours: existing wind and solar used (`we`, `se`), new wind and solar used (`wn`, `sn`), gas (`cc`, `ct`), battery charge, discharge and energy (`ch`, `dc`, `e`) and unserved demand (`u`). Options add a CO₂ cap, a carbon price, or a 300 MW load with the buyer's own wind, solar and battery (`wb`, `sb`, `chb`, `dcb`, `eb` and sizes $W_b, S_b, B_b$) under an annual or an hourly matching rule.

Rows (the constraints), each hour one hour long, so MW for that hour is MWh:

| Row | For each hour $t$ |
|---|---|
| balance | $\mathit{we} + \mathit{se} + \mathit{wn} + \mathit{sn} + \mathit{cc} + \mathit{ct} + d - c + u\ (+\, \mathit{wb} + \mathit{sb} + d^b - c^b) = D_t\ (+\,300)$ |
| sizes | $\mathit{wn} \le f^w_t W$, $\mathit{sn} \le f^s_t S$, $c, d \le B$, $e \le 4B$ |
| battery | $e_t = e_{t-1} + 0.85\,c_t - d_t$, with hour 0 of each day following its hour 23 (each day ends with the energy it began with) |
| cap | $\sum_t w_t (r_{cc}\,\mathit{cc}_t + r_{ct}\,\mathit{ct}_t) \le \varepsilon$ |
| hourly matching | $\mathit{wb} + \mathit{sb} + d^b - c^b \ge 300$ and $c^b \le \mathit{wb} + \mathit{sb}$ |
| annual matching | $\sum_t w_t (\mathit{wb}_t + \mathit{sb}_t) \ge 300 \times 8{,}760$ |

In [ ]:
H_VARS = ["we", "se", "wn", "sn", "cc", "ct", "ch", "dc", "e", "u"]
B_VARS = ["wb", "sb", "chb", "dcb", "eb"]

def build(cap=None, tau=0.0, load=0.0, buy=None):
    hv = H_VARS + (B_VARS if buy else [])
    sizes = ["W", "S", "B"] + (["Wb", "Sb", "Bb"] if buy else [])
    col = {s: i for i, s in enumerate(sizes)}; n0 = len(sizes)
    hc = lambda v, t: n0 + hv.index(v) * NH + t
    n = n0 + len(hv) * NH
    c = np.zeros(n); lo = np.zeros(n); hi = np.full(n, np.inf)
    for s in sizes:
        k = s[0]; c[col[s]] = CAP[k] + FOM[k]
        hi[col[s]] = B[{"W": "wind", "S": "solar", "B": "battery"}[k]]["max"]
    if buy == "annual":
        hi[col["Bb"]] = 0; c[col["Bb"]] = 0
    for t in range(NH):
        hi[hc("we", t)] = WE[t]; hi[hc("se", t)] = SE[t]
        hi[hc("cc", t)] = GAS["cc"]["mw"]; hi[hc("ct", t)] = GAS["ct"]["mw"]
        c[hc("cc", t)] = WGT[t] * (GAS["cc"]["cost"] + tau * GAS["cc"]["rate"]); c[hc("ct", t)] = WGT[t] * (GAS["ct"]["cost"] + tau * GAS["ct"]["rate"])
        c[hc("u", t)] = WGT[t] * M6["voll"]
    eq, ub, beq, bub, names = ([], [], []), ([], [], []), [], [], []
    def row(store, rhs, terms, b):
        r = len(rhs)
        for j, v in terms:
            store[0].append(r); store[1].append(j); store[2].append(v)
        rhs.append(b)
    for t in range(NH):
        p = (t // 24) * 24 + (t - 1) % 24
        terms = [(hc(v, t), 1) for v in ["we", "se", "wn", "sn", "cc", "ct", "dc", "u"]] + [(hc("ch", t), -1)]
        if buy:
            terms += [(hc("wb", t), 1), (hc("sb", t), 1), (hc("dcb", t), 1), (hc("chb", t), -1)]
        row(eq, beq, terms, DEM[t] + load); names.append(("b", t))
        row(eq, beq, [(hc("e", t), 1), (hc("e", p), -1), (hc("ch", t), -0.85), (hc("dc", t), 1)], 0); names.append(("sb", t))
        row(ub, bub, [(hc("wn", t), 1), (col["W"], -CFW[t])], 0); row(ub, bub, [(hc("sn", t), 1), (col["S"], -CFS[t])], 0)
        row(ub, bub, [(hc("ch", t), 1), (col["B"], -1)], 0); row(ub, bub, [(hc("dc", t), 1), (col["B"], -1)], 0); row(ub, bub, [(hc("e", t), 1), (col["B"], -4)], 0)
        if buy:
            row(ub, bub, [(hc("wb", t), 1), (col["Wb"], -CFW[t])], 0); row(ub, bub, [(hc("sb", t), 1), (col["Sb"], -CFS[t])], 0)
            if buy == "hourly":
                row(ub, bub, [(hc("wb", t), -1), (hc("sb", t), -1), (hc("dcb", t), -1), (hc("chb", t), 1)], -load)
                row(ub, bub, [(hc("chb", t), 1), (hc("wb", t), -1), (hc("sb", t), -1)], 0)
                row(ub, bub, [(hc("chb", t), 1), (col["Bb"], -1)], 0); row(ub, bub, [(hc("dcb", t), 1), (col["Bb"], -1)], 0); row(ub, bub, [(hc("eb", t), 1), (col["Bb"], -4)], 0)
                row(eq, beq, [(hc("eb", t), 1), (hc("eb", p), -1), (hc("chb", t), -0.85), (hc("dcb", t), 1)], 0); names.append(("sbb", t))
            else:
                hi[hc("chb", t)] = 0; hi[hc("dcb", t)] = 0; hi[hc("eb", t)] = 0
    cap_row = None
    if cap is not None:
        cap_row = len(bub); row(ub, bub, [(hc("cc", t), WGT[t] * GAS["cc"]["rate"]) for t in range(NH)] + [(hc("ct", t), WGT[t] * GAS["ct"]["rate"]) for t in range(NH)], cap)
    if buy == "annual":
        row(ub, bub, [(hc("wb", t), -WGT[t]) for t in range(NH)] + [(hc("sb", t), -WGT[t]) for t in range(NH)], -load * 8760)
    A_eq = coo_matrix((eq[2], (eq[0], eq[1])), shape=(len(beq), n)).tocsr(); A_ub = coo_matrix((ub[2], (ub[0], ub[1])), shape=(len(bub), n)).tocsr()
    return dict(c=c, A_eq=A_eq, b_eq=np.array(beq, float), A_ub=A_ub, b_ub=np.array(bub, float), lo=lo, hi=hi, n=n, hc=hc, col=col, cap_row=cap_row, tau=tau, buy=buy)

def solve(m, c=None, extra=None):
    A_ub, b_ub = m["A_ub"], m["b_ub"]
    if extra is not None:   # one more <= row: (coefficients, bound)
        from scipy.sparse import vstack
        A_ub = vstack([A_ub, coo_matrix(extra[0].reshape(1, -1))]).tocsr(); b_ub = np.r_[b_ub, extra[1]]
    res = linprog(m["c"] if c is None else c, A_ub=A_ub, b_ub=b_ub, A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=list(zip(m["lo"], m["hi"])), method="highs")
    assert res.status == 0, res.message
    return res

def co2_vec(m):
    v = np.zeros(m["n"])
    for t in range(NH):
        v[m["hc"]("cc", t)] = WGT[t] * GAS["cc"]["rate"]; v[m["hc"]("ct", t)] = WGT[t] * GAS["ct"]["rate"]
    return v

def report(m, res):
    x = res.x; co2 = co2_vec(m) @ x
    sizes = {k: x[i] for k, i in m["col"].items()}
    resource = sum((CAP[k[0]] + FOM[k[0]]) * v for k, v in sizes.items()) + sum(WGT[t] * (GAS["cc"]["cost"] * x[m["hc"]("cc", t)] + GAS["ct"]["cost"] * x[m["hc"]("ct", t)] + M6["voll"] * x[m["hc"]("u", t)]) for t in range(NH))
    return {"co2": co2, "cost": resource, **sizes}

m0 = build(); r0 = solve(m0); base = report(m0, r0)
print(f"LP size: {m0['n']} columns x {m0['A_eq'].shape[0] + m0['A_ub'].shape[0]} rows")
print(f"no carbon rule: CO2 {base['co2'] / 1e6:.3f} Mt, cost ${base['cost'] / 1e6:.1f}M, new wind {base['W']:.0f} MW")
assert (m0["n"], m0["A_eq"].shape[0] + m0["A_ub"].shape[0]) == (REF["size"]["cols"], REF["size"]["rows"])
assert abs(base["co2"] - REF["E0"]) < 1 and abs(base["cost"] - REF["base"]["cost"]) < 1

## 3. Caps, their duals and the frontier

The epsilon-constraint method: cap the year's CO₂ at a share of the no-rule plan's and minimize cost. `res.ineqlin.marginals` gives the change of the optimal cost per unit increase of each `<=` row's bound: for the cap it is negative, and minus it is the cap's dual in \$ per tonne, the magnitude of the frontier's downward slope.

Check it against two finite-step changes of the cost (the cap loosened and tightened by 1,000 t). Agreement is a sensitivity check, not a proof of a unique dual. If the two disagree, shrink the step and look for a corner of the frontier nearby: only different limiting one-sided slopes show the dual is not unique.

In [ ]:
E0 = base["co2"]; frontier = []
for z in REF["caps"]:
    m = build(cap=z["cap"] / 100 * E0); res = solve(m); rep = report(m, res)
    dual = -res.ineqlin.marginals[m["cap_row"]]
    d = 1000.0
    bu, bd = m["b_ub"].copy(), m["b_ub"].copy(); bu[m["cap_row"]] += d; bd[m["cap_row"]] -= d
    up = linprog(m["c"], A_ub=m["A_ub"], b_ub=bu, A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=list(zip(m["lo"], m["hi"])), method="highs").fun
    dn = linprog(m["c"], A_ub=m["A_ub"], b_ub=bd, A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=list(zip(m["lo"], m["hi"])), method="highs").fun
    print(f"cap {z['cap']}%: CO2 {rep['co2'] / 1e6:.3f} Mt, cost ${rep['cost'] / 1e6:.1f}M, dual ${dual:.2f}/t (finite steps of 1,000 t: {(res.fun - up) / d:.2f}, {(dn - res.fun) / d:.2f}), wind {rep['W']:.0f} MW, solar {rep['S']:.0f} MW")
    assert abs(rep["co2"] - z["co2"]) < 1 and abs(rep["cost"] - z["cost"]) < 1 and abs(dual - z["dual"]) < 1e-3
    assert abs((res.fun - up) / d - dual) < 1e-3 and abs((dn - res.fun) / d - dual) < 1e-3
    frontier.append((rep["co2"], rep["cost"], dual))
assert all(b[2] >= a[2] - 1e-6 for a, b in zip(frontier, frontier[1:])), "the duals rise as the cap tightens"
print("The caps and their duals match the browser.")

## 4. A price instead of a cap, and its ties

1. Drop the cap and add a price $\tau$ per tonne to every gas MWh.
2. Set $\tau$ to a cap's dual at full precision; the dual printed to the cent may not tie. The cap's plan then scores the best weighted cost, but other plans may too.
3. To find them all, hold the weighted cost within a small numerical tolerance of the optimum: a cent plus a billionth of it, about half a dollar here. A solver's optimum is exact only to its own tolerance, so demanding equality could cut off plans that tie.
4. Minimize, then maximize, CO₂ on that set. If the two differ, the price picks a stretch of the frontier, not a plan. If they agree, it picks one CO₂ outcome; the schedules behind it may still differ.

In [ ]:
def price_range(tau):
    m = build(tau=tau); res = solve(m); v = co2_vec(m)
    face = (m["c"], res.fun + 0.01 + 1e-9 * abs(res.fun))
    lo = solve(m, c=v, extra=face).fun; hi = -solve(m, c=-v, extra=face).fun
    return res, lo, hi

for z in REF["ties"]:
    res, lo, hi = price_range(z["tau"])
    print(f"price ${z['tau']:.2f}/t (the {z['cap']}% cap's dual): best plans emit {lo / 1e6:.3f} to {hi / 1e6:.3f} Mt")
    assert abs(lo - z["lo"]) < 2000 and abs(hi - z["hi"]) < 2000
for p in REF["prices"][::2]:
    res, lo, hi = price_range(p["tau"])
    assert hi - lo < 5000 and abs(lo - p["co2"]) < 5000
print("A slider price picks one CO2 outcome (at this precision); a cap's exact dual can tie a whole stretch.")

## 5. A data center and two matching rules

- **The load:** a flat 300 MW. With no rule, the LP serves it however is cheapest, and the buyer's share is counted at average grid factors.
- **Annual matching:** the buyer's own new wind and sun must make the load's year.
- **Hourly matching:** the buyer's own wind, sun and battery must cover every hour, the battery charging only from them.
- **The claim:** both rules let the buyer claim zero on the six days, under the market-based method (each certificate counted once and retired for this buyer). Section 6 checks the full year.
- **Compare** the added system cost (new plants and gas for the whole grid; the buyer's own bill depends on contracts, which the model leaves out) and what the plants emit against the grid without the load.

In [ ]:
cases = {}
for k, buy in (("none", None), ("annual", "annual"), ("hourly", "hourly")):
    m = build(load=M6["load"]["mw"], buy=buy); res = solve(m); rep = report(m, res); cases[k] = rep
    z = REF["buy"][k]
    print(f"{k:7s} CO2 {(rep['co2'] - E0) / 1e6:+.2f} Mt vs no load, cost {(rep['cost'] - cases['none']['cost']) / 1e6:+.1f}M vs no rule, "
          f"buyer: wind {rep.get('Wb', 0):.0f} MW, solar {rep.get('Sb', 0):.0f} MW, battery {rep.get('Bb', 0):.0f} MW")
    assert abs(rep["co2"] - z["co2"]) < 1 and abs(rep["cost"] - z["cost"]) < 1
    for s in ("Wb", "Sb", "Bb"):
        assert abs(rep.get(s, 0) - z[s]) < 0.01
print("Annual and hourly matching: the same claim (zero on the six days), different consequences. Certificates from existing wind change the claim, not the plan: their plan is 'none'.")

## 6. The full year

Six days stood for 2025. Check two results on all 8,760 hours: does the annual buyer's wind make the load's year, and how much energy must the hourly portfolio leave unmatched at the least (its battery cyclic over the year)? The least unmatched energy is unique; which hours carry it can differ between equally good schedules, so count energy, not hours.

In [ ]:
Wb = cases["annual"]["Wb"]; share = PW.sum() * Wb / (300 * 8760)
print(f"annual: {Wb:.0f} MW of wind make {share:.1%} of the load's year on the full year")
assert abs(share - REF["fullYear"]["annualShare"]) < 1e-6

h = cases["hourly"]; N = 8760; nv = 6
i = np.arange(N); prev = (i - 1) % N
c = np.zeros(nv * N); c[5 * N:] = 1                  # columns: w, s, ch, dc, e, short
lo = np.zeros(nv * N); hi = np.concatenate([PW * h["Wb"], PS * h["Sb"], np.full(N, h["Bb"]), np.full(N, h["Bb"]), np.full(N, 4 * h["Bb"]), np.full(N, np.inf)])
rows, cols, vals = [], [], []
def add(r, cc, v):
    rows.extend(r); cols.extend(cc); vals.extend(v)
# matched: w + s + dc - ch + short >= 300  ->  -(...) <= -300 ; charge only from own wind and sun: ch - w - s <= 0
add(i, i, -np.ones(N)); add(i, N + i, -np.ones(N)); add(i, 3 * N + i, -np.ones(N)); add(i, 2 * N + i, np.ones(N)); add(i, 5 * N + i, -np.ones(N))
add(N + i, 2 * N + i, np.ones(N)); add(N + i, i, -np.ones(N)); add(N + i, N + i, -np.ones(N))
A_ub = coo_matrix((vals, (rows, cols)), shape=(2 * N, nv * N)).tocsr(); b_ub = np.r_[np.full(N, -300.0), np.zeros(N)]
A_eq = coo_matrix((np.r_[np.ones(N), -np.ones(N), np.full(N, -0.85), np.ones(N)], (np.r_[i, i, i, i], np.r_[4 * N + i, 4 * N + prev, 2 * N + i, 3 * N + i])), shape=(N, nv * N)).tocsr()
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=np.zeros(N), bounds=list(zip(lo, hi)), method="highs")
assert res.status == 0, res.message
short = res.x[5 * N:]
print(f"hourly: at least {res.fun:,.0f} MWh unmatched in 2025 ({res.fun / (300 * N):.3%} of the load); this schedule puts it in {int((short > 1e-6).sum())} hours of low wind and sun")
assert abs(res.fun - REF["fullYear"]["shortMWh"]) < 1
print("The lab matches the browser.")

## 7. Practice (about 90 minutes)

Write your own cells below each task. Predict first, then solve; state every result against a named baseline.

1. **A finer frontier** (about 15 minutes). Solve caps every 2% from 90% to 70%. Where does the dual jump, and what changes in the plan there?
2. **Ties in the schedule** (about 15 minutes). For the 50% cap, test whether the total energy curtailed differs between best plans: hold the cost within section 4's tolerance, then minimize and maximize it. Do the same for the energy curtailed at the existing wind farms alone. Which of the two can differ, and why? Check your four numbers with `check_task2` below.
   - Curtailed energy: what the weather allows (the existing farms' hourly output, plus each new size times its capacity factor) minus what is used, each hour weighted by its days, in MWh a year.
3. **A price sweep** (about 10 minutes). Sweep the price from \$20 to \$24 per t in steps of \$0.10. Plot CO₂ against the price and explain the jump.
4. **More days** (about 20 minutes). Add two more typical days per season (the next closest to each medoid) and re-weight. How much do the frontier and the hourly portfolio move?
5. **Not deliverable** (about 15 minutes). Let the buyer's annual matching be met by wind on another grid: drop its output from this grid's balance. What do the plants emit now?
6. **Write it up** (about 15 minutes). In five sentences, report what hourly matching costs here, what it changes at the plants, against which baseline, and what the six days cannot show.

In [ ]:
# Task 2's check: your four numbers, in MWh a year, against the reference block (which the lesson's tests solve afresh).
def check_task2(total_least, total_most, existing_wind_least, existing_wind_most, tol=5):
    z = REF["practice"]["cap50"]
    for name, mine, ref in (("total, least", total_least, z["curtailed"][0]), ("total, most", total_most, z["curtailed"][1]),
                            ("existing wind, least", existing_wind_least, z["existingWindCurtailed"][0]),
                            ("existing wind, most", existing_wind_most, z["existingWindCurtailed"][1])):
        assert abs(mine - ref) <= tol, f"{name}: {mine:,.0f} MWh is not the lesson's; check how you hold the cost and count the curtailed energy"
    print("Task 2 matches the lesson's best plans.")

assert REF["practice"]["cap50"]["cap"] == 50
# check_task2(total_least, total_most, existing_wind_least, existing_wind_most)